# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This playbook combines two things validated in earlier weeks: the client-held-out Gradient
Boosting model from w05/w06 (decline probability per page) and the CTR-vs-position-tier rule
from w04 (still useful as a second, independent signal, not replaced by the model).

**Archetypes, in priority order for review:**

| Archetype | Condition | Action | Reason code |
|---|---|---|---|
| `refresh_priority` | model decline probability >= 0.6, impressions >= 500 | `refresh_priority_review` | `high_risk_high_demand` |
| `ctr_opportunity` | CTR < 50% of tier benchmark, impressions >= 500, not already refresh_priority | `review_for_ctr_fix` | `ctr_underperform_vs_position` |
| `monitor_low_demand` | model decline probability >= 0.6, impressions < 500 | `monitor_low_priority` | `high_risk_low_demand` |
| `protect` | position_tier in {top_3, page_1}, decline probability < 0.3 | `protect_distribute` | `strong_stable_asset` |
| `monitor_routine` | everything else | `monitor_routine` | *(none)* |

**Decay/refresh insight** (grounded in this week's own data, not just the paper): pages in the
`91-180` day freshness bucket carry the highest average model-predicted decline probability
(0.623) of any freshness bucket with real sample size, versus 0.524 for freshly-updated (`0-30`
day) pages. This echoes the MIXED staleness signal from w04, now visible in the model's own
output rather than just the rule -- consistent enough across two different methods to treat as a
real, if moderate, pattern worth acting on.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

features = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
            "content_age_days", "days_since_last_update", "word_count",
            "engagement_rate", "scroll_rate"]

# same validated model + split as w05/w06
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(eligible, groups=eligible["client_id"]))
train = eligible.iloc[train_idx]
model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model.fit(train[features].fillna(0), train["is_declining_label"])
eligible["decline_probability"] = model.predict_proba(eligible[features].fillna(0))[:, 1]

# the w04 CTR-vs-tier rule, rebuilt identically
tier_mean_ctr = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True)["ctr"].mean()
eligible["tier_mean_ctr"] = eligible["position_tier"].map(tier_mean_ctr)

# decay/refresh insight, grounded in this week's data
fresh_insight = eligible.groupby("freshness_tier", observed=True)["decline_probability"].agg(["mean", "count"])
print("decline probability by freshness tier:")
print(fresh_insight)

# archetype assignment, in priority order
def assign_archetype(row):
    if row["decline_probability"] >= 0.6 and row["impressions_90d"] >= 500:
        return "refresh_priority", "refresh_priority_review", "high_risk_high_demand"
    if row["ctr"] < 0.5 * row["tier_mean_ctr"] and row["impressions_90d"] >= 500:
        return "ctr_opportunity", "review_for_ctr_fix", "ctr_underperform_vs_position"
    if row["decline_probability"] >= 0.6 and row["impressions_90d"] < 500:
        return "monitor_low_demand", "monitor_low_priority", "high_risk_low_demand"
    if row["position_tier"] in ("top_3", "page_1") and row["decline_probability"] < 0.3:
        return "protect", "protect_distribute", "strong_stable_asset"
    return "monitor_routine", "monitor_routine", ""

results = eligible.apply(assign_archetype, axis=1, result_type="expand")
eligible["archetype"], eligible["action"], eligible["reason_code"] = results[0], results[1], results[2]

# playbook score: only actionable archetypes get a positive ranking score
actionable = {"refresh_priority", "ctr_opportunity", "monitor_low_demand"}
eligible["playbook_score"] = np.where(
    eligible["archetype"].isin(actionable),
    eligible["decline_probability"] * np.log1p(eligible["impressions_90d"]),
    0.0,
)
eligible["playbook_score"] = 100 * eligible["playbook_score"] / eligible["playbook_score"].max()

print("\narchetype counts:")
print(eligible["archetype"].value_counts())

queue = eligible.sort_values("playbook_score", ascending=False)[
    ["content_id", "client_id", "position_tier", "impressions_90d", "decline_probability",
     "ctr", "tier_mean_ctr", "freshness_tier", "archetype", "action", "reason_code", "playbook_score"]
].reset_index(drop=True)
queue.head(10)


decline probability by freshness tier:
                    mean  count
freshness_tier                 
0-30            0.523526  20480
181+            0.493308    174
31-90           0.733963    175
91-180          0.622997   9171



archetype counts:
archetype
monitor_routine       10239
refresh_priority       9043
monitor_low_demand     5715
protect                2676
ctr_opportunity        2327
Name: count, dtype: int64


,content_id,client_id,position_tier,impressions_90d,decline_probability,ctr,tier_mean_ctr,freshness_tier,archetype,action,reason_code,playbook_score
0,content_370de6e8e035,client_7f2253d7e2,page_3_5,114389,0.945140,0.13,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,100.000000
1,content_7158cfbbc450,client_7f2253d7e2,page_3_5,134567,0.930709,0.06,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,99.846594
2,content_551fe371f51b,client_7f2253d7e2,page_3_5,115789,0.923356,0.04,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,97.797144
3,content_66b4046cc144,client_7f2253d7e2,page_3_5,217415,0.874538,0.03,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,97.631747
4,content_41b9311ed497,client_7f2253d7e2,page_3_5,65669,0.952548,0.10,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,95.981684
5,content_150f89b1d73b,client_7f2253d7e2,page_3_5,83490,0.923557,0.04,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,95.074713
6,content_a7c2dfc8a6ec,client_7f2253d7e2,page_3_5,76868,0.927241,0.01,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,94.757987
7,content_b51e2e4d22ff,client_7f2253d7e2,page_3_5,91795,0.909646,0.04,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,94.426284
8,content_c84a0ab98e90,client_f369cb89fc,page_1,223271,0.838381,0.03,0.354760,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,93.797734
9,content_2333ccd359f7,client_7f2253d7e2,page_3_5,43280,0.942408,0.14,0.142359,0-30,refresh_priority,refresh_priority_review,high_risk_high_demand,91.390723


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content editor or SEO lead with limited review time per sprint uses this
queue to decide which pages to look at *first*, out of a much larger inventory. It orders
candidates by a combination of demand (impressions) and modeled decline risk, and gives each one
a stated reason so the reviewer isn't just trusting a number blindly.

**Where it stops being valid:**
- The label the model was trained on (`trend_direction == "down"`) is a same-window proxy, not a
  verified future outcome (flagged since w02) -- the queue tells you "this page currently looks
  like it's declining by one particular definition," not "this page will keep declining."
- Validated on 32 clients from one dataset snapshot. It has not been tested on a client outside
  this dataset, a different industry vertical, or a different time period.
- The model's own feature importances (from w05) lean heavily on `impressions_90d` and
  `content_age_days` (57% combined) rather than content-quality signals -- so a "high risk" score
  is partly just "this page is old and/or has a lot of traffic," which a reviewer should weigh
  accordingly, not treat as a diagnosis of *why* a page is struggling.
- This is decision-support for *prioritizing review*, not a verdict that any specific page needs
  a specific fix, and it does not predict whether a refresh will actually recover traffic (that
  would need a real experiment, not this data).


In [2]:
# Show the split between "actionable" and "not actionable" volume, to make the limits concrete.
actionable_share = eligible["archetype"].isin(actionable).mean()
print(f"share of eligible pages the playbook actively recommends reviewing: {actionable_share:.1%}")
print(f"share left as protect/monitor_routine (no action this cycle): {1 - actionable_share:.1%}")

# how much of the "risk" signal is coming from volume/age vs quality, per w05's own feature importances
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
volume_age_share = importances[["impressions_90d", "content_age_days"]].sum()
print(f"\nshare of model importance from volume/age alone: {volume_age_share:.1%} (context for the limits above)")


share of eligible pages the playbook actively recommends reviewing: 57.0%
share left as protect/monitor_routine (no action this cycle): 43.0%

share of model importance from volume/age alone: 57.4% (context for the limits above)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `refresh_priority` or `ctr_opportunity` row, a human must check:**
1. Does the page still match real user intent, or has the topic moved on? (No feature here
   measures intent match.)
2. Is a competitor's rich result or featured snippet the actual reason for low CTR -- a rewrite
   won't fix that.
3. Is this page a `consolidation` case (a sibling page absorbed its traffic) rather than a real
   decline? The lane guide flags this exact look-alike pattern; nothing in this pipeline checks
   for it.
4. Is the low CTR/low volume actually a tracking or attribution gap, not a real user behavior?

**What should NOT be automated:**
- Auto-publishing any content change based on this score alone -- every action here needs a human
  editor's judgment call, every time.
- Auto-deprioritizing/pruning a page purely because it scored `monitor_routine` -- this queue
  ranks review urgency, not page value; a `monitor_routine` page could still matter strategically.
- Treating `reason_code` as a diagnosis to copy into a ticket without checking the underlying
  numbers -- it's a pointer to *what to look at*, not a finished explanation.
- Using this to make claims about Google's algorithm or to promise a refresh will "cause" a
  traffic recovery -- neither claim is supported by this data (same limit as every earlier week).


In [3]:
# Make the no-go boundary concrete: show how many top-ranked pages fall into look-alike risk
# categories a human still needs to check (using signals available in this dataset as a proxy).
top_100 = queue.head(100)
low_volume_edge = (top_100["impressions_90d"] < 700).sum()
print(f"of the top 100 ranked pages, {low_volume_edge} sit close to the volume floor (<700 impressions)")
print("-- these are the ones most worth a human sanity check before committing edit time.")


of the top 100 ranked pages, 0 sit close to the volume floor (<700 impressions)
-- these are the ones most worth a human sanity check before committing edit time.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Data drift check:** if the eligible pool's average `impressions_90d` or `content_age_days`
  shifts more than ~20% from this run's values, the model's feature distributions no longer match
  what it was trained on -- retrain before trusting new scores.
- **Precision check:** re-run this playbook's `refresh_priority` queue against real outcomes each
  quarter (did the flagged pages actually decline further, or recover after review?). If
  Precision@50 drops meaningfully below the 0.76 validated in w05/w06, that's a retrain trigger,
  not just a bad quarter.
- **Archetype balance check:** if `monitor_routine` share drifts sharply up or down from the
  baseline below, the underlying content mix has changed enough that the thresholds
  (0.6 decline probability, 500-impression floor, etc.) may need revisiting rather than assuming
  they still fit.
- **New client onboarding:** any new client added to the review pool should be treated as
  out-of-distribution until the model has been checked specifically against that client's data --
  the validation in w05/w06 only covers the 32 clients in this snapshot.


In [4]:
baseline_archetype_mix = eligible["archetype"].value_counts(normalize=True)
print("archetype mix at this run -- the baseline future runs should be compared against:")
print(baseline_archetype_mix)


archetype mix at this run -- the baseline future runs should be compared against:
archetype
monitor_routine       0.341300
refresh_priority      0.301433
monitor_low_demand    0.190500
protect               0.089200
ctr_opportunity       0.077567
Name: proportion, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


In [5]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# full ranked queue -- gitignored by design, regenerated on every run
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"wrote {len(queue):,} rows to work/outputs/action_playbook_queue.csv")

# figure: archetype mix, reused in the paper's Results/Recommendations section
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
eligible["archetype"].value_counts().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("pages")
ax.set_title("Action playbook: archetype mix (eligible pages)")
plt.tight_layout()
fig.savefig("work/figures/archetype_mix.png", dpi=150)
plt.close(fig)
print("wrote work/figures/archetype_mix.png")

# figure: decline probability by freshness tier (the decay/refresh insight)
fig2, ax2 = plt.subplots(figsize=(7, 4))
fresh_insight["mean"].plot(kind="bar", ax=ax2, color="#DD8452")
ax2.set_ylabel("mean decline probability")
ax2.set_title("Decline probability by freshness tier")
plt.tight_layout()
fig2.savefig("work/figures/decline_by_freshness.png", dpi=150)
plt.close(fig2)
print("wrote work/figures/decline_by_freshness.png")

# JSON receipt -- this one IS committed
summary = {
    "model": "GradientBoostingClassifier (w05/w06 validated, client-grouped split)",
    "precision_at_50_validated": 0.76,
    "n_eligible": int(len(eligible)),
    "archetype_counts": eligible["archetype"].value_counts().to_dict(),
    "actionable_share": float(eligible["archetype"].isin(actionable).mean()),
    "decline_probability_by_freshness_tier": fresh_insight["mean"].round(3).to_dict(),
    "queue_path": "work/outputs/action_playbook_queue.csv",
    "figures": ["work/figures/archetype_mix.png", "work/figures/decline_by_freshness.png"],
}
with open("work/outputs/w07_playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("wrote work/outputs/w07_playbook_summary.json")
summary


wrote 30,000 rows to work/outputs/action_playbook_queue.csv


wrote work/figures/archetype_mix.png


wrote work/figures/decline_by_freshness.png
wrote work/outputs/w07_playbook_summary.json


{'model': 'GradientBoostingClassifier (w05/w06 validated, client-grouped split)',
 'precision_at_50_validated': 0.76,
 'n_eligible': 30000,
 'archetype_counts': {'monitor_routine': 10239,
  'refresh_priority': 9043,
  'monitor_low_demand': 5715,
  'protect': 2676,
  'ctr_opportunity': 2327},
 'actionable_share': 0.5695,
 'decline_probability_by_freshness_tier': {'0-30': 0.524,
  '181+': 0.493,
  '31-90': 0.734,
  '91-180': 0.623},
 'queue_path': 'work/outputs/action_playbook_queue.csv',
 'figures': ['work/figures/archetype_mix.png',
  'work/figures/decline_by_freshness.png']}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.